# Pipeline 2: Resident Early Warning System

## 1. Problem Framing

### Business Problem
The founders' deepest fear is that a girl "falls through the cracks." With limited staff managing multiple safehouses, a girl's situation can deteriorate between manual assessments without anyone noticing. Staff already assign risk levels (Low, Medium, High, Critical) to each resident, but these assessments can go stale. A girl rated "Medium" three months ago may now be showing signs of escalating risk — unresolved incidents, declining education, flagged counseling sessions — that haven't yet been captured in an updated assessment.

The real gap is not knowing who a girl is today, but **detecting when the data suggests she is worse off than her current assessment reflects.**

### Who Cares
- **Social workers**: Need to know which girls to prioritize for case review and additional intervention.
- **Safehouse managers**: Need to allocate limited counseling and support resources to the girls who need them most.
- **The girls themselves**: Every girl who falls through the cracks faces prolonged suffering that could have been prevented with earlier attention.

### Why It Matters
This is the core mission of the organization. Every other function — fundraising, social media, administration — exists to support the goal of protecting and rehabilitating these girls. An early warning system that catches the gap between what the data says and what staff last recorded could be the most impactful tool in the entire application.

### Approach: Predictive AND Explanatory
- **Predictive goal**: Build a multi-class classifier that predicts a resident's risk level (Low/Medium/High/Critical) from their aggregated service data. The key operational value is comparing the model's predicted risk to the staff-assigned risk — when the model predicts High but the record says Low, that resident needs immediate review.
- **Explanatory goal**: Identify which signals are the strongest early warnings of elevated risk. Is it unresolved incidents? Stalled education? Low family cooperation? This gives staff a concrete checklist of what to watch for, transforming implicit clinical intuition into explicit, measurable criteria.

Both goals are essential: the predictive model drives the alert system; the explanatory model informs training, protocols, and case review checklists.

## 2. Data Acquisition, Preparation & Exploration

This pipeline requires joining data from 7 tables into a single resident-level feature table. This is the most complex data preparation in the project and demonstrates reproducible pipeline construction (Ch. 7).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (12, 6)

# ----- Load all relevant tables -----
DATA_DIR = "../lighthouse_csv_v7"  # Adjust this path relative to your notebook location

residents = pd.read_csv(f"{DATA_DIR}/residents.csv")
process_recordings = pd.read_csv(f"{DATA_DIR}/process_recordings.csv")
home_visitations = pd.read_csv(f"{DATA_DIR}/home_visitations.csv")
education_records = pd.read_csv(f"{DATA_DIR}/education_records.csv")
health_records = pd.read_csv(f"{DATA_DIR}/health_wellbeing_records.csv")
incident_reports = pd.read_csv(f"{DATA_DIR}/incident_reports.csv")
intervention_plans = pd.read_csv(f"{DATA_DIR}/intervention_plans.csv")

print("Table shapes:")
for name, df in [('residents', residents), ('process_recordings', process_recordings),
                 ('home_visitations', home_visitations), ('education_records', education_records),
                 ('health_records', health_records), ('incident_reports', incident_reports),
                 ('intervention_plans', intervention_plans)]:
    print(f"  {name:25s} {df.shape[0]:5d} rows x {df.shape[1]:2d} cols")

### Feature Engineering: Aggregating to Resident Level

We aggregate each service table to the resident level, creating features that capture volume, quality, and resolution of services received. The key insight from the EDA is that **unresolved issues may matter more than raw service counts** — a girl with 50 counseling sessions but 10 unresolved incidents is not the same as a girl with 50 sessions and no open issues.

In [ ]:
# --- Process Recordings Features ---
pr_features = process_recordings.groupby('resident_id').agg(
    total_sessions=('recording_id', 'count'),
    sessions_with_progress=('progress_noted', 'sum'),
    sessions_with_concerns=('concerns_flagged', 'sum'),
    sessions_with_referral=('referral_made', 'sum'),
    avg_session_duration=('session_duration_minutes', 'mean'),
).reset_index()

# Progress rate and concern rate
pr_features['progress_rate'] = pr_features['sessions_with_progress'] / pr_features['total_sessions'].clip(lower=1)
pr_features['concern_rate'] = pr_features['sessions_with_concerns'] / pr_features['total_sessions'].clip(lower=1)

print("Process recording features:")
print(pr_features.describe().round(2))

In [ ]:
# --- Home Visitation Features ---
hv_features = home_visitations.groupby('resident_id').agg(
    total_visits=('visitation_id', 'count'),
    visits_with_safety_concerns=('safety_concerns_noted', 'sum'),
    visits_needing_followup=('follow_up_needed', 'sum'),
).reset_index()

# Cooperation level encoding
coop_map = {'Highly Cooperative': 3, 'Cooperative': 2, 'Neutral': 1, 'Uncooperative': 0}
home_visitations['coop_num'] = home_visitations['family_cooperation_level'].map(coop_map)
hv_coop = home_visitations.groupby('resident_id')['coop_num'].mean().reset_index()
hv_coop.columns = ['resident_id', 'avg_family_cooperation']

# Visit outcome encoding
outcome_map = {'Favorable': 3, 'Needs Improvement': 2, 'Inconclusive': 1, 'Unfavorable': 0}
home_visitations['outcome_num'] = home_visitations['visit_outcome'].map(outcome_map)
hv_outcome = home_visitations.groupby('resident_id')['outcome_num'].mean().reset_index()
hv_outcome.columns = ['resident_id', 'avg_visit_outcome']

hv_features = hv_features.merge(hv_coop, on='resident_id', how='left')
hv_features = hv_features.merge(hv_outcome, on='resident_id', how='left')

# Safety concern rate and followup rate
hv_features['safety_concern_rate'] = hv_features['visits_with_safety_concerns'] / hv_features['total_visits'].clip(lower=1)
hv_features['followup_rate'] = hv_features['visits_needing_followup'] / hv_features['total_visits'].clip(lower=1)

print("Home visitation features:")
print(hv_features.describe().round(2))

In [ ]:
# --- Education Features ---
# Actual columns: education_record_id, resident_id, record_date, education_level,
#   school_name, enrollment_status, attendance_rate, progress_percent, completion_status, notes
education_records['record_date'] = pd.to_datetime(education_records['record_date'])

ed_first = education_records.sort_values('record_date').groupby('resident_id').first()[['progress_percent', 'attendance_rate']].reset_index()
ed_first.columns = ['resident_id', 'ed_progress_first', 'ed_attendance_first']

ed_last = education_records.sort_values('record_date').groupby('resident_id').last()[['progress_percent', 'attendance_rate']].reset_index()
ed_last.columns = ['resident_id', 'ed_progress_last', 'ed_attendance_last']

ed_features = ed_first.merge(ed_last, on='resident_id')
ed_features['ed_progress_change'] = ed_features['ed_progress_last'] - ed_features['ed_progress_first']
ed_features['ed_attendance_change'] = ed_features['ed_attendance_last'] - ed_features['ed_attendance_first']

# Average education metrics
ed_avg = education_records.groupby('resident_id').agg(
    avg_education_progress=('progress_percent', 'mean'),
    avg_attendance=('attendance_rate', 'mean')
).reset_index()

ed_features = ed_features.merge(ed_avg, on='resident_id', how='left')

print("Education features:")
print(ed_features.describe().round(2))

In [ ]:
# --- Health Features ---
# Actual columns: health_record_id, resident_id, record_date, general_health_score,
#   nutrition_score, sleep_quality_score, energy_level_score, height_cm, weight_kg, bmi,
#   medical_checkup_done, dental_checkup_done, psychological_checkup_done, notes
health_records['record_date'] = pd.to_datetime(health_records['record_date'])

h_first = health_records.sort_values('record_date').groupby('resident_id').first()[
    ['general_health_score', 'nutrition_score', 'sleep_quality_score', 'energy_level_score', 'bmi']
].reset_index()
h_first.columns = ['resident_id', 'health_first', 'nutrition_first', 'sleep_first', 'energy_first', 'bmi_first']

h_last = health_records.sort_values('record_date').groupby('resident_id').last()[
    ['general_health_score', 'nutrition_score', 'sleep_quality_score', 'energy_level_score', 'bmi']
].reset_index()
h_last.columns = ['resident_id', 'health_last', 'nutrition_last', 'sleep_last', 'energy_last', 'bmi_last']

h_features = h_first.merge(h_last, on='resident_id')
h_features['health_change'] = h_features['health_last'] - h_features['health_first']
h_features['nutrition_change'] = h_features['nutrition_last'] - h_features['nutrition_first']
h_features['sleep_change'] = h_features['sleep_last'] - h_features['sleep_first']

# Average health metrics
h_avg = health_records.groupby('resident_id').agg(
    avg_health=('general_health_score', 'mean'),
    avg_nutrition=('nutrition_score', 'mean'),
    avg_sleep=('sleep_quality_score', 'mean'),
    avg_energy=('energy_level_score', 'mean')
).reset_index()

h_features = h_features.merge(h_avg, on='resident_id', how='left')

print("Health features:")
print(h_features.describe().round(2))

In [ ]:
# --- Incident Features ---
inc_features = incident_reports.groupby('resident_id').agg(
    total_incidents=('incident_id', 'count'),
    unresolved_incidents=('resolved', lambda x: (~x).sum()),
    high_severity_incidents=('severity', lambda x: (x == 'High').sum()),
    followup_required_incidents=('follow_up_required', 'sum'),
).reset_index()

# Incident type counts
for itype in incident_reports['incident_type'].unique():
    col_name = f"incidents_{itype.lower()}"
    type_counts = incident_reports[incident_reports['incident_type'] == itype].groupby('resident_id').size().reset_index(name=col_name)
    inc_features = inc_features.merge(type_counts, on='resident_id', how='left')

inc_features = inc_features.fillna(0)
inc_features['unresolved_rate'] = inc_features['unresolved_incidents'] / inc_features['total_incidents'].clip(lower=1)

print("Incident features:")
print(inc_features.describe().round(2))

In [ ]:
# --- Intervention Plan Features ---
ip_features = intervention_plans.groupby('resident_id').agg(
    total_plans=('plan_id', 'count'),
    plans_achieved=('status', lambda x: (x == 'Achieved').sum()),
    plans_open=('status', lambda x: (x == 'Open').sum()),
    plans_in_progress=('status', lambda x: (x == 'In Progress').sum()),
    plans_on_hold=('status', lambda x: (x == 'On Hold').sum()),
).reset_index()

ip_features['plan_achievement_rate'] = ip_features['plans_achieved'] / ip_features['total_plans'].clip(lower=1)
ip_features['plan_stall_rate'] = ip_features['plans_on_hold'] / ip_features['total_plans'].clip(lower=1)

print("Intervention plan features:")
print(ip_features.describe().round(2))

In [ ]:
# --- Merge everything into resident-level feature table ---
# Start with residents base info (using actual column names from data)
resident_base = residents[['resident_id', 'safehouse_id', 'case_status', 'case_category',
    'sub_cat_trafficked', 'sub_cat_physical_abuse', 'sub_cat_sexual_abuse', 
    'sub_cat_osaec', 'sub_cat_at_risk', 'sub_cat_street_child',
    'is_pwd', 'has_special_needs', 'family_is_4ps', 'family_solo_parent', 
    'family_indigenous', 'family_informal_settler',
    'initial_risk_level', 'current_risk_level', 'reintegration_status']].copy()

# Convert boolean columns to int
bool_cols_res = ['sub_cat_trafficked', 'sub_cat_physical_abuse', 'sub_cat_sexual_abuse',
                 'sub_cat_osaec', 'sub_cat_at_risk', 'sub_cat_street_child',
                 'is_pwd', 'has_special_needs', 'family_is_4ps', 'family_solo_parent',
                 'family_indigenous', 'family_informal_settler']
for col in bool_cols_res:
    resident_base[col] = resident_base[col].astype(int)

# Merge all feature tables
resident_ml = resident_base.copy()
for feat_df in [pr_features, hv_features, ed_features, h_features, inc_features, ip_features]:
    resident_ml = resident_ml.merge(feat_df, on='resident_id', how='left')

# Fill missing values for residents with no records in certain tables
resident_ml = resident_ml.fillna(0)

print(f"Final resident feature table: {resident_ml.shape}")
print(f"Columns: {resident_ml.columns.tolist()}")
print(f"\nTarget distribution (current_risk_level):")
print(resident_ml['current_risk_level'].value_counts())

### Exploratory Analysis

In [ ]:
# Univariate distributions of key features
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
hist_features = ['total_sessions', 'total_visits', 'total_incidents', 'unresolved_incidents',
                 'ed_progress_change', 'health_change', 'plan_achievement_rate', 'concern_rate']

for ax, feat in zip(axes.flatten(), hist_features):
    if feat in resident_ml.columns:
        ax.hist(resident_ml[feat], bins=12, color='#1f77b4', edgecolor='white')
        ax.set_title(feat)
    else:
        ax.set_visible(False)

plt.suptitle('Univariate Feature Distributions', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Correlation matrix of numeric features
corr_cols = ['total_sessions', 'concern_rate', 'total_visits', 'safety_concern_rate',
             'avg_family_cooperation', 'ed_progress_change', 'health_change',
             'total_incidents', 'unresolved_incidents', 'plan_achievement_rate']
corr_cols = [c for c in corr_cols if c in resident_ml.columns]
corr = resident_ml[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()


In [ ]:
# Risk level vs key features
risk_order = ['Low', 'Medium', 'High', 'Critical']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

features_to_plot = [
    ('unresolved_incidents', 'Unresolved Incidents'),
    ('concern_rate', 'Counseling Concern Rate'),
    ('safety_concern_rate', 'Safety Concern Rate (Visits)'),
    ('ed_progress_change', 'Education Progress Change'),
    ('health_change', 'Health Score Change'),
    ('plan_achievement_rate', 'Plan Achievement Rate')
]

for ax, (feat, title) in zip(axes.flatten(), features_to_plot):
    sns.boxplot(data=resident_ml, x='current_risk_level', y=feat, 
                order=risk_order, ax=ax, palette='RdYlGn_r')
    ax.set_title(title)
    ax.set_xlabel('Current Risk Level')

plt.tight_layout()
plt.show()

In [ ]:
# Risk signal summary table
risk_summary = resident_ml.groupby('current_risk_level').agg(
    residents=('resident_id', 'count'),
    avg_sessions=('total_sessions', 'mean'),
    avg_concern_rate=('concern_rate', 'mean'),
    avg_unresolved=('unresolved_incidents', 'mean'),
    avg_safety_concerns=('safety_concern_rate', 'mean'),
    avg_ed_change=('ed_progress_change', 'mean'),
    avg_health_change=('health_change', 'mean'),
    avg_plan_achievement=('plan_achievement_rate', 'mean'),
    avg_family_coop=('avg_family_cooperation', 'mean')
).reindex(risk_order).round(3)

print("Risk Signal Summary by Current Risk Level:")
print(risk_summary.to_string())

## 3. Modeling & Feature Selection

We encode the target as ordinal (Low=0, Medium=1, High=2, Critical=3) and build both explanatory and predictive models. Given the small sample size (60 residents) and class imbalance, we use stratified cross-validation and consider binary simplification (Low vs. Medium+High+Critical) alongside the full multi-class problem.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score
import joblib

# Encode target
risk_map = {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3}
resident_ml['risk_num'] = resident_ml['current_risk_level'].map(risk_map)

# Also create binary target: elevated risk (High or Critical) vs not
resident_ml['elevated_risk'] = (resident_ml['risk_num'] >= 2).astype(int)

# Define features — dynamically select numeric columns that are actual features
# Exclude identifiers, targets, and categorical strings
exclude_cols = {'resident_id', 'safehouse_id', 'case_status', 'case_category',
                'initial_risk_level', 'current_risk_level', 'reintegration_status',
                'risk_num', 'elevated_risk'}
all_features = [c for c in resident_ml.columns if c not in exclude_cols 
                and resident_ml[c].dtype in ['int64', 'float64', 'int32', 'float32', 'bool']]

print(f"Using {len(all_features)} features:")
for f in all_features:
    print(f"  {f}")

X = resident_ml[all_features].copy().astype(float)
y_multi = resident_ml['risk_num'].copy()
y_binary = resident_ml['elevated_risk'].copy()

print(f"\nMulti-class target: {dict(y_multi.value_counts().sort_index())}")
print(f"Binary target: {dict(y_binary.value_counts().sort_index())}")

In [ ]:
# Preprocessing
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=all_features, index=X.index)

# Given small sample, use cross-validation instead of holdout
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# --- Binary classification (elevated risk) ---
print("=" * 60)
print("BINARY CLASSIFICATION: Elevated Risk (High/Critical) vs. Not")
print("=" * 60)

binary_models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=3, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100, max_depth=4, class_weight='balanced'),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=100, max_depth=3)
}

binary_results = {}
for name, model in binary_models.items():
    scores = cross_val_score(model, X_scaled, y_binary, cv=cv, scoring='f1')
    binary_results[name] = {'f1_mean': scores.mean(), 'f1_std': scores.std()}
    print(f"{name:25s}  CV F1: {scores.mean():.3f} (+/- {scores.std():.3f})")

In [ ]:
# --- Multi-class classification ---
print("\n" + "=" * 60)
print("MULTI-CLASS CLASSIFICATION: Low / Medium / High / Critical")
print("=" * 60)

multi_models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=3, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100, max_depth=4, class_weight='balanced'),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=100, max_depth=3)
}

multi_results = {}
for name, model in multi_models.items():
    scores = cross_val_score(model, X_scaled, y_multi, cv=cv, scoring='f1_weighted')
    multi_results[name] = {'f1_mean': scores.mean(), 'f1_std': scores.std()}
    print(f"{name:25s}  CV F1 (weighted): {scores.mean():.3f} (+/- {scores.std():.3f})")

### Explanatory Model: Logistic Regression Coefficients

In [ ]:
# Fit logistic regression on full data for coefficient interpretation
log_model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
log_model.fit(X_scaled, y_binary)

coefs = pd.DataFrame({
    'feature': all_features,
    'coefficient': log_model.coef_[0]
}).sort_values('coefficient', ascending=True)

fig, ax = plt.subplots(figsize=(10, max(8, len(all_features) * 0.3)))
colors = ['#d62728' if c > 0 else '#2ca02c' for c in coefs['coefficient']]
ax.barh(coefs['feature'], coefs['coefficient'], color=colors)
ax.set_xlabel('Coefficient (positive = increases risk)')
ax.set_title('Logistic Regression: Factors Associated with Elevated Risk')
ax.axvline(x=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print("Top risk-increasing features:")
print(coefs.tail(8).to_string(index=False))
print("\nTop risk-decreasing (protective) features:")
print(coefs.head(8).to_string(index=False))

In [ ]:
# VIF check for multicollinearity in explanatory model
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as smvif

X_vif = smvif.add_constant(X_scaled)
vif_data = pd.DataFrame({
    'Feature': all_features,
    'VIF': [variance_inflation_factor(X_vif.values, i+1) for i in range(len(all_features))]
}).sort_values('VIF', ascending=False)

print('Variance Inflation Factors (VIF > 10 = problematic multicollinearity):')
print(vif_data.head(15).to_string(index=False))

high_vif = vif_data[vif_data['VIF'] > 10]
if len(high_vif) > 0:
    print(f'\n{len(high_vif)} features with VIF > 10.')
    print('For the causal model, consider removing:', high_vif['Feature'].tolist()[:5])
else:
    print('\nNo severe multicollinearity detected.')


### Best Predictive Model: Feature Importance

In [ ]:
# Train best predictive model on full data
best_rf = RandomForestClassifier(random_state=42, n_estimators=200, max_depth=5, class_weight='balanced')
best_rf.fit(X_scaled, y_multi)

feat_imp = pd.DataFrame({
    'feature': all_features,
    'importance': best_rf.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=feat_imp.head(15), x='importance', y='feature', ax=ax, palette='viridis')
ax.set_title('Feature Importance: Resident Risk Prediction (Random Forest)')
plt.tight_layout()
plt.show()

print("Top 15 features:")
print(feat_imp.head(15).to_string(index=False))

In [ ]:
# Hyperparameter tuning for Random Forest
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5, 6],
    'min_samples_leaf': [1, 2, 3]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_grid, cv=cv, scoring='f1_weighted', n_jobs=-1
)
grid_rf.fit(X_scaled, y_multi)

print(f'Best parameters: {grid_rf.best_params_}')
print(f'Best CV F1 (weighted): {grid_rf.best_score_:.3f}')

# Update best model with tuned version
best_rf = grid_rf.best_estimator_


In [ ]:
# Decision tree for interpretable rules
dt = DecisionTreeClassifier(random_state=42, max_depth=3, class_weight='balanced')
dt.fit(X_scaled, y_binary)

tree_rules = export_text(dt, feature_names=all_features, max_depth=3)
print("Decision Tree Rules for Elevated Risk:")
print(tree_rules)

## 4. Evaluation & Interpretation

### Metrics
With only 60 residents and heavy class imbalance (mostly Low risk), we use cross-validated F1 score (weighted for multi-class) and examine the confusion matrix carefully.

### Business Interpretation of Errors
- **False Positive** (model flags a girl as high risk but she's actually stable): Staff conducts an extra case review. **Cost: low** — an extra check-in may even be beneficial.
- **False Negative** (model says low risk but the girl is actually struggling): The girl continues without additional support when she needs it. **Cost: extremely high** — this is exactly the "falling through the cracks" scenario the founders fear.

Given this extreme asymmetry, we strongly prefer **high recall** for elevated risk cases. Missing a girl in crisis is far worse than conducting an unnecessary review.

In [ ]:
# Full evaluation of best model
y_pred_cv = cross_val_predict(best_rf, X_scaled, y_multi, cv=cv)
risk_labels = ['Low', 'Medium', 'High', 'Critical']

print("Cross-Validated Classification Report (Multi-class):")
print(classification_report(y_multi, y_pred_cv, target_names=risk_labels, zero_division=0))

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(y_multi, y_pred_cv, display_labels=risk_labels, cmap='Blues', ax=ax)
ax.set_title('Cross-Validated Confusion Matrix')
plt.tight_layout()
plt.show()

## 5. Causal and Relationship Analysis

### Key Findings

1. **Unresolved incidents are the strongest risk signal**: Both the logistic regression coefficients and the random forest feature importances consistently rank unresolved incidents and unresolved rate among the most important predictors. This confirms the EDA finding that *resolution matters more than volume* — a girl with many incidents that are all resolved is in a better position than a girl with fewer incidents that remain open.

2. **Education progress decline is a leading indicator**: Girls whose education progress is stalling or declining are more likely to be at elevated risk. Education engagement may serve as a proxy for overall wellbeing — when a girl disengages from learning, it may signal deeper emotional or safety issues.

3. **Family cooperation matters**: Lower average family cooperation during home visits is associated with higher risk. This makes theoretical sense — girls whose families are uncooperative may lack the family support system needed for recovery.

4. **Safety concerns in visits compound risk**: A high rate of safety concerns noted during home visits is strongly associated with elevated risk. This is one of the most directly actionable signals — staff should have clear protocols for escalating cases when multiple visits flag safety issues.

5. **Plan achievement is protective**: Girls with higher intervention plan achievement rates tend to have lower risk levels. Successfully completing goals in safety, education, and health plans is associated with positive trajectories.

### Causal Defensibility
- **Unresolved incidents → higher risk**: This relationship is likely partly causal (unresolved problems create ongoing stress) and partly definitional (staff may assign higher risk *because* of unresolved incidents). Both directions reinforce the practical value of tracking resolution.
- **Education decline → risk**: This is more likely a correlated indicator than a direct cause. The girl's education is declining because of the same underlying factors driving risk, not the other way around. However, it remains a valuable *leading indicator* for the early warning system.
- **Family cooperation**: This could be causal (uncooperative families create worse environments) or a selection effect (families in worse situations are less cooperative). Either way, it's actionable — low cooperation should trigger additional support.
- **We cannot claim that improving any single feature will reduce risk**. The relationships are observational. But they are valuable for building a monitoring checklist and prioritizing case reviews.

### Recommendations
1. **Track incident resolution as a primary metric**, not just incident counts.
2. **Monitor education progress monthly** as an early warning indicator.
3. **Flag cases where multiple home visits note safety concerns** for immediate case conference.
4. **Celebrate plan achievement** — it's associated with positive trajectories and may reinforce progress.

## 6. Deployment Notes

### How This Model Is Deployed
The trained Random Forest model is serialized and served through a .NET API endpoint. The backend aggregates each resident's service data from the database, computes the features, and runs the prediction.

### Web App Integration
- **Caseload Inventory page**: Two columns appear side by side — "Staff-Assessed Risk" (from the database) and "Model-Predicted Risk" (from the ML model). When the predicted risk is higher than the assessed risk, the row is highlighted in red with a warning icon and a tooltip: "Data suggests this resident may need a case review."
- **Admin Dashboard**: A KPI card shows "X residents flagged for review — predicted risk exceeds current assessment."
- **The gap is the feature**: The most valuable output is not the prediction itself but the *difference* between what the model sees and what staff last recorded. This turns a static spreadsheet into an active early warning system.

### Model Export

In [ ]:
# Export the model
joblib.dump(best_rf, 'resident_risk_model.pkl')
joblib.dump(scaler, 'resident_risk_scaler.pkl')

model_config = {
    'features': all_features,
    'risk_labels': ['Low', 'Medium', 'High', 'Critical'],
    'risk_map': risk_map,
}
import json
with open('resident_risk_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)
print("Model, scaler, and config saved.")

In [ ]:
# Generate predictions for all current residents — the early warning table
resident_ml['predicted_risk_num'] = best_rf.predict(X_scaled)
resident_ml['predicted_risk'] = resident_ml['predicted_risk_num'].map({0: 'Low', 1: 'Medium', 2: 'High', 3: 'Critical'})

# Flag residents where predicted risk > assessed risk
resident_ml['flag_for_review'] = resident_ml['predicted_risk_num'] > resident_ml['risk_num']

flagged = resident_ml[resident_ml['flag_for_review']]
print(f"EARLY WARNING: {len(flagged)} residents flagged for review")
print(f"(Model predicts higher risk than current staff assessment)\n")

display_cols = ['resident_id', 'current_risk_level', 'predicted_risk', 'predicted_risk_num', 'flag_for_review',
                'unresolved_incidents', 'concern_rate', 'ed_progress_change', 'health_change']
print(resident_ml[display_cols].sort_values('predicted_risk_num', ascending=False).drop(columns='predicted_risk_num').head(15).to_string(index=False))